In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import ollama

# ----- Load dataset -----
df = pd.read_excel("clean_QA.xlsx")

# ----- Embedding model -----
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
question_embeddings = embed_model.encode(df["question"].tolist(), convert_to_tensor=True)

# ----- Helper: summarize long context using mistral -----
def summarize_context(text):
    prompt = f"Please summarize the following medical information into 3–5 concise sentences:\n\n{text}\n\nSummary:"
    response = ollama.generate(model='mistral', prompt=prompt)
    return response["response"].strip()

# ----- Helper: generate final answer -----
def generate_answer(question, context_summary):
    prompt = f"""
You are a medical assistant AI. Use the following summarized information to answer the user's question clearly and helpfully.

Summary Information:
{context_summary}

User Question:
{question}

Final Answer:
"""
    response = ollama.generate(model='mistral', prompt=prompt)
    return response["response"].strip()

# ----- Chatbot -----
def chatbot(question):

    # Step 1: RAG retrieval
    q_emb = embed_model.encode(question, convert_to_tensor=True)
    scores = util.cos_sim(q_emb, question_embeddings)[0]
    best_idx = scores.argmax().item()
    best_score = scores[best_idx].item()

    # Step 2: If match good → use RAG + summarization + generation
    if best_score >= 0.45:
        source_answer = df.iloc[best_idx]["answer"]

        # summarize Excel answer first (to avoid long content)
        summary = summarize_context(source_answer)

        # generate final response
        return generate_answer(question, summary)

    else:
        # Step 3: fallback for general health questions
        prompt = f"Answer this health question clearly and accurately:\n{question}"
        response = ollama.generate(model='mistral', prompt=prompt)
        return response["response"].strip()


# ----- Chat Loop -----
while True:
    q = input("Ask a health question: ")
    print("\nAI Answer:", chatbot(q))
    print("-" * 60)



/opt/anaconda3/envs/healthbot/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/opt/anaconda3/envs/healthbot/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/opt/anaconda3/envs/healthbot/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File

Ask a health question: How many people are affected by Balance Problems ?

AI Answer: Approximately 14.8% of American adults experienced balance or dizziness issues in the year 2008. However, it's important to note that this data is from a decade ago, and prevalence rates may have changed since then. For more current statistics, I would recommend consulting a recent study or a reliable health organization's website.
------------------------------------------------------------
Ask a health question: How many people are affected by Heart Failure ?

AI Answer: Approximately 5 million individuals in the U.S. are affected by heart failure.
------------------------------------------------------------
Ask a health question: How many people are affected by Osteoarthritis ?

AI Answer: Approximately one-third (33.6%) of individuals aged 65 and above are affected by Osteoarthritis, according to the provided summary information. However, without more specific data for other age groups or populati

KeyboardInterrupt: Interrupted by user